In [5]:
from vllm import LLM

In [ ]:
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
from typing import List, Dict, Optional, Tuple
import torch
import numpy as np

class SimpleVLLMInference:
    def __init__(self, model_id: str, tensor_parallel_size: int = 1, max_logprobs: int = 5000):
        """Simple vLLM inference setup for token-based generation"""
        self.model_id = model_id
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        
        # Initialize vLLM model with custom max_logprobs
        self.llm = LLM(
            model=model_id,
            trust_remote_code=True,
            tensor_parallel_size=tensor_parallel_size,
            gpu_memory_utilization=0.95,
            max_logprobs=max_logprobs  # Override the default limit of 20
        )
        
        # Define decoding strategies (similar to commons.py)
        self.decoding_configs = {
            "greedy": {
                "temperature": 0,
                "top_p": 1.0,
                "max_tokens": 128
            },
            "nucleus": {
                "temperature": 1.0,
                "top_p": 0.9,
                "max_tokens": 128
            }
        }
    
    def get_sampling_params(self, strategy: str = "greedy", **kwargs) -> SamplingParams:
        """Get sampling parameters for a specific decoding strategy"""
        if strategy not in self.decoding_configs:
            raise ValueError(f"Unknown strategy: {strategy}. Choose from {list(self.decoding_configs.keys())}")
        
        # Start with base config for the strategy
        config = self.decoding_configs[strategy].copy()
        
        # Override with any custom parameters
        config.update(kwargs)
        
        # Create SamplingParams
        return SamplingParams(**config)
    
    def generate_from_tokens(
        self, 
        token_ids: List[List[int]], 
        strategy: str = "greedy",
        **kwargs
    ) -> Tuple[List[str], List[List[int]]]:
        """
        Generate responses from token IDs with specified decoding strategy
        
        Args:
            token_ids: List of token ID sequences to use as prompts
            strategy: Decoding strategy ('greedy', 'nucleus')
            **kwargs: Override sampling parameters
            
        Returns:
            - List of generated texts
            - List of generated token IDs
        """
        
        # Get sampling parameters for the strategy
        sampling_params = self.get_sampling_params(strategy, **kwargs)
        
        # Generate with vLLM
        outputs = self.llm.generate(
            prompt_token_ids=token_ids,
            sampling_params=sampling_params,
            use_tqdm=True
        )
        
        # Sort outputs by request ID
        outputs = sorted(outputs, key=lambda x: int(x.request_id))
        
        # Extract generated text and token IDs
        texts = [output.outputs[0].text for output in outputs]
        generated_token_ids = [output.outputs[0].token_ids for output in outputs]
        
        return texts, generated_token_ids
    
    def calc_reference_nll(self, prefix_suffix_tokens: List[List[int]], suffix_length: int) -> List[Dict]:
        """
        Calculate NLL for REFERENCE (gold) suffix - mimics calc_reference_nll from commons.py
        Uses raw model probabilities (temperature=1.0) to match PyTorch's CrossEntropyLoss
        
        Args:
            prefix_suffix_tokens: Full sequences with gold suffix (prefix + true suffix)
            suffix_length: Length of the suffix to calculate metrics for
            
        Returns:
            List of dictionaries with ref_nll_mean, ref_nll_std, and ref_perplexity
        """
        results = []
        
        for seq in prefix_suffix_tokens:
            # Add BOS token if needed (like commons.py does)
            bos_token_id = self.tokenizer.bos_token_id if self.tokenizer.bos_token_id else 1
            seq_with_bos = [bos_token_id] + seq
            
            # Get logprobs for the entire sequence
            # IMPORTANT: Use temperature=1.0 to get raw logprobs (equivalent to commons.py raw logits)
            # temperature=0 would give modified logprobs for greedy mode
            sampling_params = SamplingParams(
                temperature=1.0,  # Get raw log_softmax(logits), not greedy-modified
                max_tokens=1,  # We don't want to generate
                prompt_logprobs=len(seq_with_bos),
                logprobs=1
            )
            
            outputs = self.llm.generate(
                prompt_token_ids=[seq_with_bos],
                sampling_params=sampling_params,
                use_tqdm=False
            )
            
            prompt_logprobs = outputs[0].prompt_logprobs
            
            # Extract suffix NLLs (matching commons.py logic)
            suffix_nlls = []
            if prompt_logprobs and len(prompt_logprobs) > 1:
                # Calculate suffix start position (accounting for BOS)
                suffix_start = max(1, len(seq_with_bos) - suffix_length)
                end_idx = len(seq_with_bos)
                
                for i in range(suffix_start, end_idx):
                    if i < len(prompt_logprobs) and prompt_logprobs[i]:
                        token_id = seq_with_bos[i]
                        if token_id in prompt_logprobs[i]:
                            # NLL = -log(p)
                            suffix_nlls.append(-prompt_logprobs[i][token_id].logprob)
            
            # Calculate metrics using numpy (matching commons.py)
            if suffix_nlls:
                nlls_array = np.array(suffix_nlls)
                ref_nll_mean = np.mean(nlls_array)
                ref_nll_std = np.std(nlls_array) if len(nlls_array) > 1 else 0.0
                ref_perplexity = np.exp(ref_nll_mean)
            else:
                ref_nll_mean = float('inf')
                ref_nll_std = 0.0
                ref_perplexity = float('inf')
            
            results.append({
                'ref_nll_mean': float(ref_nll_mean),
                'ref_nll_std': float(ref_nll_std),
                'ref_perplexity': float(ref_perplexity)
            })
        
        return results
    
    def calc_generation_nll(self, complete_sequences: List[List[int]], suffix_length: int) -> List[Dict]:
        """
        Calculate NLL for GENERATED suffix - mimics calc_generation_nll_forward from commons.py
        Uses raw model probabilities (temperature=1.0) to match PyTorch's CrossEntropyLoss
        
        Args:
            complete_sequences: Full sequences (BOS + prefix + generated suffix)
            suffix_length: Length of the suffix to calculate metrics for
            
        Returns:
            List of dictionaries with nll_mean, nll_std, and perplexity
        """
        results = []
        
        for seq in complete_sequences:
            # Sequence should already have BOS token
            # IMPORTANT: Use temperature=1.0 to get raw logprobs (equivalent to commons.py raw logits)
            # This measures how likely the generated sequence is under the model's learned distribution
            sampling_params = SamplingParams(
                temperature=1.0,  # Get raw log_softmax(logits), not greedy-modified
                max_tokens=1,  # We don't want to generate
                prompt_logprobs=len(seq),
                logprobs=1
            )
            
            outputs = self.llm.generate(
                prompt_token_ids=[seq],
                sampling_params=sampling_params,
                use_tqdm=False
            )
            
            prompt_logprobs = outputs[0].prompt_logprobs
            
            # Extract suffix NLLs (matching commons.py logic)
            suffix_nlls = []
            if prompt_logprobs and len(prompt_logprobs) > 1:
                # Calculate suffix start position
                suffix_start = max(1, len(seq) - suffix_length)
                end_idx = len(seq)
                
                for i in range(suffix_start, end_idx):
                    if i < len(prompt_logprobs) and prompt_logprobs[i]:
                        token_id = seq[i]
                        if token_id in prompt_logprobs[i]:
                            # NLL = -log(p)
                            suffix_nlls.append(-prompt_logprobs[i][token_id].logprob)
            
            # Calculate metrics using numpy (matching commons.py)
            if suffix_nlls:
                nlls_array = np.array(suffix_nlls)
                nll_mean = np.mean(nlls_array)
                nll_std = np.std(nlls_array) if len(nlls_array) > 1 else 0.0
                perplexity = np.exp(nll_mean)
            else:
                nll_mean = float('inf')
                nll_std = 0.0
                perplexity = float('inf')
            
            results.append({
                'nll_mean': float(nll_mean),
                'nll_std': float(nll_std),
                'perplexity': float(perplexity)
            })
        
        return results

In [ ]:
# Example with NLL calculations (matching commons.py workflow)
model_id = "/capstor/store/cscs/swissai/infra01/swiss-alignment/checkpoints/Apertus3-8B_iter_1678000-tulu3-sft/checkpoint-13446/"

# Initialize with tensor_parallel_size=2 for 8B model
inference = SimpleVLLMInference(model_id, tensor_parallel_size=2, max_logprobs=5000)

# Example: prefix-suffix with reference text (like commons.py)
prefix_text = "The capital of Switzerland"
reference_suffix = " is Bern, a beautiful city"
suffix_length = 10  # For generation

# Tokenize prefix and full sequence
prefix_tokens = inference.tokenizer.encode(prefix_text)
full_tokens = inference.tokenizer.encode(prefix_text + reference_suffix)
actual_suffix_length = len(full_tokens) - len(prefix_tokens)

print(f"Prefix: '{prefix_text}'")
print(f"Reference suffix: '{reference_suffix}'")
print(f"Prefix tokens: {len(prefix_tokens)}")
print(f"Full sequence tokens: {len(full_tokens)}")
print(f"Actual suffix tokens: {actual_suffix_length}")

# Calculate reference NLL (for the gold/true suffix)
print("\n" + "="*60)
print("REFERENCE NLL CALCULATION")
print("="*60)

ref_nll_results = inference.calc_reference_nll(
    [full_tokens],  # Full sequence with true suffix
    suffix_length=actual_suffix_length
)
print(f"Reference NLL mean: {ref_nll_results[0]['ref_nll_mean']:.4f}")
print(f"Reference NLL std: {ref_nll_results[0]['ref_nll_std']:.4f}")
print(f"Reference perplexity: {ref_nll_results[0]['ref_perplexity']:.2f}")

# Generate and calculate generation NLL
print("\n" + "="*60)
print("GENERATION AND NLL CALCULATION")
print("="*60)

# Add BOS token for generation
bos_id = inference.tokenizer.bos_token_id if inference.tokenizer.bos_token_id else 1
input_with_bos = [bos_id] + prefix_tokens

# Generate with greedy strategy
texts, gen_tokens = inference.generate_from_tokens(
    [input_with_bos],
    strategy="greedy",
    max_tokens=suffix_length,
    min_tokens=suffix_length
)

print(f"Generated text: '{texts[0]}'")
print(f"Generated tokens: {gen_tokens[0][:10]}...")

# Calculate NLL for generated sequence
complete_sequence = input_with_bos + gen_tokens[0]
gen_nll_results = inference.calc_generation_nll(
    [complete_sequence],
    suffix_length=suffix_length
)

print(f"\nGeneration NLL mean: {gen_nll_results[0]['nll_mean']:.4f}")
print(f"Generation NLL std: {gen_nll_results[0]['nll_std']:.4f}")
print(f"Generation perplexity: {gen_nll_results[0]['perplexity']:.2f}")

# Compare different strategies
print("\n" + "="*60)
print("COMPARING STRATEGIES WITH NLL")
print("="*60)

strategies = ["greedy", "nucleus"]
for strategy in strategies:
    texts, gen_tokens = inference.generate_from_tokens(
        [input_with_bos],
        strategy=strategy,
        max_tokens=suffix_length,
        min_tokens=suffix_length
    )
    
    complete_seq = input_with_bos + gen_tokens[0]
    nll_results = inference.calc_generation_nll([complete_seq], suffix_length)
    
    print(f"\n{strategy.upper()}:")
    print(f"  Text: '{texts[0][:50]}...'")
    print(f"  Perplexity: {nll_results[0]['perplexity']:.2f}")

In [8]:
# Import the existing functions from commons.py
import sys
sys.path.append('/iopsstor/scratch/cscs/xyixuan/PDM/src')

from infer.commons import batch_processing_gutenberg, process_dataset

# Load Gutenberg dataset using existing functions
data_path = "/iopsstor/scratch/cscs/xyixuan/dataset/gutenberg_apertus_buk167/rep_128_token.jsonl"
prefix_length = 500
suffix_length = 500
offset = 0

# Process dataset using the existing function
dataset = process_dataset(
    data_path=data_path,
    batch_processing_fn=batch_processing_gutenberg,
    prefix_length=prefix_length,
    suffix_length=suffix_length,
    offset=offset,
    num_proc=4  # Number of processes for parallel processing
)

print(f"Loaded {len(dataset)} samples from Gutenberg dataset")

# Test with a few samples
num_test_samples = 3
for i in range(min(num_test_samples, len(dataset))):
    sample = dataset[i]
    print(f"\nSample {i+1}:")
    print(f"  Total tokens: {len(sample)}")
    print(f"  Prefix tokens: {prefix_length}")
    print(f"  Suffix tokens: {suffix_length}")
    
    # Decode to see the text
    prefix_text = inference.tokenizer.decode(sample[:prefix_length])
    suffix_text = inference.tokenizer.decode(sample[prefix_length:])
    print(f"  Prefix preview: '{prefix_text[:100]}...'")
    print(f"  Suffix preview: '{suffix_text[:100]}...'")

Generating train split: 167 examples [00:00, 13260.05 examples/s]
Generating prefix and suffix pairs (num_proc=4): 100%|██████████| 167/167 [00:00<00:00, 793.62 examples/s]

Loaded 167 samples from Gutenberg dataset

Sample 1:
  Total tokens: 1000
  Prefix tokens: 500
  Suffix tokens: 500
  Prefix preview: ', and that of the street confined
between the houses. In one of the buildings crouching in the shado...'
  Suffix preview: ' walk of one of the Paris cemeteries one bright, warm morning,
while around them stretched the garde...'

Sample 2:
  Total tokens: 1000
  Prefix tokens: 500
  Suffix tokens: 500
  Prefix preview: ' of those mixed types, unharmonious, common among
mongrel races. Her black hair shone like jet, her ...'
  Suffix preview: 'went back to the Maltese, doubtless because her conversation was more
diverting and spicy.



_THE C...'

Sample 3:
  Total tokens: 1000
  Prefix tokens: 500
  Suffix tokens: 500
  Prefix preview: 'to let your voice rise or fall--it does not matter which--on the phrase
"in every way."  This is per...'
  Suffix preview: ' times of health it may be regarded as an
envoy going before to clear the path of whatever evils 

In [9]:
# Calculate NLL for Gutenberg samples
print("\n" + "="*60)
print("NLL CALCULATION ON GUTENBERG SAMPLES")
print("="*60)

for i in range(min(3, len(dataset))):
    sample = dataset[i]
    
    print(f"\n--- Sample {i+1} ---")
    
    # Calculate reference NLL (for the gold suffix in the dataset)
    ref_nll_results = inference.calc_reference_nll(
        [sample],  # Full sequence with true suffix
        suffix_length=suffix_length
    )
    
    print(f"Reference NLL mean: {ref_nll_results[0]['ref_nll_mean']:.4f}")
    print(f"Reference perplexity: {ref_nll_results[0]['ref_perplexity']:.2f}")
    
    # Generate with greedy and calculate generation NLL
    bos_id = inference.tokenizer.bos_token_id if inference.tokenizer.bos_token_id else 1
    prefix_with_bos = [bos_id] + sample[:prefix_length]
    
    texts, gen_tokens = inference.generate_from_tokens(
        [prefix_with_bos],
        strategy="greedy",
        max_tokens=suffix_length,
        min_tokens=suffix_length
    )
    
    # Calculate NLL for generated sequence
    complete_generated = prefix_with_bos + gen_tokens[0]
    gen_nll_results = inference.calc_generation_nll(
        [complete_generated],
        suffix_length=suffix_length
    )
    
    print(f"Generation NLL mean: {gen_nll_results[0]['nll_mean']:.4f}")
    print(f"Generation perplexity: {gen_nll_results[0]['perplexity']:.2f}")
    
    # Compare original vs generated suffix
    original_suffix = inference.tokenizer.decode(sample[prefix_length:prefix_length+suffix_length])
    generated_suffix = texts[0]
    
    print(f"\nOriginal suffix (first 100 chars): '{original_suffix[:100]}...'")
    print(f"Generated suffix (first 100 chars): '{generated_suffix[:100]}...'")


NLL CALCULATION ON GUTENBERG SAMPLES

--- Sample 1 ---


/tmp/ipykernel_169553/3474320487.py:12: DeprecationWarning: The keyword arguments {'prompt_token_ids'} are deprecated and will be removed in a future update. Please use the 'prompts' parameter instead.
  ref_nll_results = inference.calc_reference_nll(
/tmp/ipykernel_169553/3474320487.py:24: DeprecationWarning: The keyword arguments {'prompt_token_ids'} are deprecated and will be removed in a future update. Please use the 'prompts' parameter instead.
  texts, gen_tokens = inference.generate_from_tokens(


Reference NLL mean: 2.6289
Reference perplexity: 13.86


Processed prompts: 100%|██████████| 1/1 [00:06<00:00,  6.31s/it, est. speed input: 79.41 toks/s, output: 79.25 toks/s]
/tmp/ipykernel_169553/3474320487.py:33: DeprecationWarning: The keyword arguments {'prompt_token_ids'} are deprecated and will be removed in a future update. Please use the 'prompts' parameter instead.
  gen_nll_results = inference.calc_generation_nll(


Generation NLL mean: 0.1679
Generation perplexity: 1.18

Original suffix (first 100 chars): ' walk of one of the Paris cemeteries one bright, warm morning,
while around them stretched the garde...'
Generated suffix (first 100 chars): ' corner of the cemetery of the Holy Innocents, and he had
kissed her again in the little garden of t...'

--- Sample 2 ---
Reference NLL mean: 2.4416
Reference perplexity: 11.49


Processed prompts: 100%|██████████| 1/1 [00:06<00:00,  6.30s/it, est. speed input: 79.47 toks/s, output: 79.31 toks/s]


Generation NLL mean: 0.2221
Generation perplexity: 1.25

Original suffix (first 100 chars): 'went back to the Maltese, doubtless because her conversation was more
diverting and spicy.



_THE C...'
Generated suffix (first 100 chars): 'escaped from them, and went back to the Marchesa Sciacca.

Laura, who was not at all fond of the Mar...'

--- Sample 3 ---
Reference NLL mean: 2.2190
Reference perplexity: 9.20


Processed prompts: 100%|██████████| 1/1 [00:06<00:00,  6.31s/it, est. speed input: 79.38 toks/s, output: 79.23 toks/s]


Generation NLL mean: 0.5055
Generation perplexity: 1.66

Original suffix (first 100 chars): ' times of health it may be regarded as an
envoy going before to clear the path of whatever evils may...'
Generated suffix (first 100 chars): ' the course of time you will find that
you can repeat it without any effort, and that it will be rep...'


In [10]:
# Test the corrected NLL calculations with different strategies
print("\n" + "="*60)
print("TESTING CORRECTED NLL CALCULATIONS")
print("="*60)

# Use a sample from the dataset
test_sample = dataset[0]
bos_id = inference.tokenizer.bos_token_id if inference.tokenizer.bos_token_id else 1
prefix_with_bos = [bos_id] + test_sample[:prefix_length]

print(f"\nTest configuration:")
print(f"  Prefix length: {prefix_length}")
print(f"  Suffix length: {suffix_length}")

# 1. Calculate reference NLL (gold suffix)
print("\n--- Reference (Gold) Suffix NLL ---")
ref_nll = inference.calc_reference_nll([test_sample], suffix_length)
print(f"  NLL mean: {ref_nll[0]['ref_nll_mean']:.4f}")
print(f"  NLL std: {ref_nll[0]['ref_nll_std']:.4f}")
print(f"  Perplexity: {ref_nll[0]['ref_perplexity']:.2f}")

# 2. Test with different generation strategies
strategies = ["greedy", "nucleus"]
for strategy in strategies:
    print(f"\n--- {strategy.upper()} Generation ---")
    
    # Generate suffix
    texts, gen_tokens = inference.generate_from_tokens(
        [prefix_with_bos],
        strategy=strategy,
        max_tokens=suffix_length,
        min_tokens=suffix_length
    )
    
    # Calculate NLL for generated sequence
    complete_seq = prefix_with_bos + gen_tokens[0]
    gen_nll = inference.calc_generation_nll([complete_seq], suffix_length)
    
    print(f"  Generation NLL mean: {gen_nll[0]['nll_mean']:.4f}")
    print(f"  Generation NLL std: {gen_nll[0]['nll_std']:.4f}")
    print(f"  Generation perplexity: {gen_nll[0]['perplexity']:.2f}")
    
    # Show first 50 chars of generated text
    print(f"  Generated text preview: '{texts[0][:50]}...'")

print("\n" + "="*60)
print("Note: All NLL calculations now use temperature=1.0 to get raw")
print("model probabilities, matching commons.py's CrossEntropyLoss approach")
print("="*60)


TESTING CORRECTED NLL CALCULATIONS

Test configuration:
  Prefix length: 500
  Suffix length: 500

--- Reference (Gold) Suffix NLL ---


/tmp/ipykernel_169553/383650999.py:17: DeprecationWarning: The keyword arguments {'prompt_token_ids'} are deprecated and will be removed in a future update. Please use the 'prompts' parameter instead.
  ref_nll = inference.calc_reference_nll([test_sample], suffix_length)
/tmp/ipykernel_169553/383650999.py:28: DeprecationWarning: The keyword arguments {'prompt_token_ids'} are deprecated and will be removed in a future update. Please use the 'prompts' parameter instead.
  texts, gen_tokens = inference.generate_from_tokens(


  NLL mean: 2.6289
  NLL std: 2.6376
  Perplexity: 13.86

--- GREEDY Generation ---


Processed prompts: 100%|██████████| 1/1 [00:06<00:00,  6.31s/it, est. speed input: 79.45 toks/s, output: 79.29 toks/s]
/tmp/ipykernel_169553/383650999.py:37: DeprecationWarning: The keyword arguments {'prompt_token_ids'} are deprecated and will be removed in a future update. Please use the 'prompts' parameter instead.
  gen_nll = inference.calc_generation_nll([complete_seq], suffix_length)


  Generation NLL mean: 0.1679
  Generation NLL std: 0.4849
  Generation perplexity: 1.18
  Generated text preview: ' corner of the cemetery of the Holy Innocents, and...'

--- NUCLEUS Generation ---


Processed prompts: 100%|██████████| 1/1 [00:06<00:00,  6.37s/it, est. speed input: 78.66 toks/s, output: 78.50 toks/s]


  Generation NLL mean: 2.2568
  Generation NLL std: 2.1623
  Generation perplexity: 9.55
  Generated text preview: ' corner of the cemetery, the walls of which opened...'

Note: All NLL calculations now use temperature=1.0 to get raw
model probabilities, matching commons.py's CrossEntropyLoss approach
